# Target encoding: five-fold validation


In [2]:
from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

In [3]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

processed_data_dir = project_root / "data" / "processed"
raw_data_dir = project_root / "data" / "raw"
oof_dir = project_root / "artifacts" / "oof"
test_predictions_dir = project_root / "artifacts" / "test_predictions"
metrics_dir = project_root / "artifacts" / "metrics"
submissions_dir = project_root / "submissions"

for directory in [oof_dir, test_predictions_dir, metrics_dir, submissions_dir]:
    directory.mkdir(parents=True, exist_ok=True)

train_df = pd.read_parquet(processed_data_dir / "train_with_folds.parquet")
test_df = pd.read_parquet(processed_data_dir / "test_clean.parquet")
sample_submission = pd.read_csv(raw_data_dir / "sample_submission.csv")

print("train:", train_df.shape)
print("test:", test_df.shape)

train: (699635, 25)
test: (299844, 23)


## Configuration


In [4]:
target = "satisfaction"
drop_columns = ["id", target, "fold"]
feature_columns = [
    column for column in train_df.columns if column not in drop_columns
]
te_columns = [
    column for column in feature_columns if column != "Arrival Delay Missing"
]

folds_to_run = [0, 1, 2, 3, 4]

previous_fold_scores = {
    0: 0.9603531995583718,
    1: 0.9588368924864543,
    2: 0.9608820613035495,
    3: 0.9603249147336925,
    4: 0.9602052661607016,
}

parameters = {
    "n_estimators": 4000,
    "learning_rate": 0.03,
    "num_leaves": 127,
    "min_child_samples": 50,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.3,
    "reg_lambda": 1.0,
    "n_jobs": 4,
    "verbosity": -1,
}

fold_metrics_path = metrics_dir / "proper_te_folds.json"

if fold_metrics_path.exists():
    with open(fold_metrics_path, encoding="utf-8") as file:
        fold_metrics = json.load(file)
else:
    fold_metrics = {}

print("base features:", len(feature_columns))
print("target-encoded features:", len(te_columns))
print("folds to run:", folds_to_run)

base features: 22
target-encoded features: 21
folds to run: [0, 1, 2, 3, 4]


## Fold preparation


In [5]:
def prepare_fold(fold):
    train_mask = train_df["fold"] != fold
    valid_mask = train_df["fold"] == fold

    x_train = train_df.loc[train_mask, feature_columns].copy()
    x_valid = train_df.loc[valid_mask, feature_columns].copy()
    x_test = test_df[feature_columns].copy()

    y_train = train_df.loc[train_mask, target]
    y_valid = train_df.loc[valid_mask, target]

    inner_folds = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42 + fold,
    )

    encoder = TargetEncoder(
        target_type="binary",
        smooth="auto",
        cv=inner_folds,
    )

    encoded_train = encoder.fit_transform(
        x_train[te_columns].astype(str),
        y_train,
    ).astype("float32")

    encoded_valid = encoder.transform(
        x_valid[te_columns].astype(str)
    ).astype("float32")

    encoded_test = encoder.transform(
        x_test[te_columns].astype(str)
    ).astype("float32")

    te_names = ["te_" + column for column in te_columns]

    x_train = pd.concat([
        x_train,
        pd.DataFrame(encoded_train, columns=te_names, index=x_train.index),
    ], axis=1)

    x_valid = pd.concat([
        x_valid,
        pd.DataFrame(encoded_valid, columns=te_names, index=x_valid.index),
    ], axis=1)

    x_test = pd.concat([
        x_test,
        pd.DataFrame(encoded_test, columns=te_names, index=x_test.index),
    ], axis=1)

    del encoded_train, encoded_valid, encoded_test, encoder
    gc.collect()

    arrival_column = "Arrival Delay in Minutes"
    arrival_median = x_train[arrival_column].median()

    for frame in [x_train, x_valid, x_test]:
        frame[arrival_column] = frame[arrival_column].fillna(arrival_median)

    class_categories = sorted(
        set(x_train["Class"].dropna().unique())
        | set(x_valid["Class"].dropna().unique())
        | set(x_test["Class"].dropna().unique())
    )

    for frame in [x_train, x_valid, x_test]:
        frame["Class"] = pd.Categorical(
            frame["Class"],
            categories=class_categories,
        )

    return x_train, x_valid, x_test, y_train, y_valid, valid_mask

## Training


In [6]:
for fold in folds_to_run:
    print(f"\nfold {fold}")

    x_train, x_valid, x_test, y_train, y_valid, valid_mask = prepare_fold(fold)

    model = lgb.LGBMClassifier(
        **parameters,
        random_state=42 + fold,
    )

    model.fit(
        x_train,
        y_train,
        eval_set=[(x_valid, y_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(150),
            lgb.log_evaluation(100),
        ],
    )

    valid_prediction = model.predict_proba(
        x_valid,
        num_iteration=model.best_iteration_,
    )[:, 1].astype("float32")

    test_prediction = model.predict_proba(
        x_test,
        num_iteration=model.best_iteration_,
    )[:, 1].astype("float32")

    fold_auc = roc_auc_score(y_valid, valid_prediction)
    difference = fold_auc - previous_fold_scores[fold]

    np.save(oof_dir / f"proper_te_fold_{fold}.npy", valid_prediction)
    np.save(
        test_predictions_dir / f"proper_te_fold_{fold}.npy",
        test_prediction,
    )

    fold_metrics[str(fold)] = {
        "auc": float(fold_auc),
        "previous_auc": float(previous_fold_scores[fold]),
        "difference": float(difference),
        "best_iteration": int(model.best_iteration_),
    }

    with open(fold_metrics_path, "w", encoding="utf-8") as file:
        json.dump(fold_metrics, file, indent=4, ensure_ascii=False)

    print("AUC:", fold_auc)
    print("difference:", difference)
    print("best iteration:", model.best_iteration_)
    print("saved fold:", fold)

    del model
    del x_train, x_valid, x_test
    del y_train, y_valid, valid_mask
    del valid_prediction, test_prediction
    gc.collect()


fold 0


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.958323	valid_0's binary_logloss: 0.235313
[200]	valid_0's auc: 0.960029	valid_0's binary_logloss: 0.226092
[300]	valid_0's auc: 0.960416	valid_0's binary_logloss: 0.224988
[400]	valid_0's auc: 0.960504	valid_0's binary_logloss: 0.224726
[500]	valid_0's auc: 0.960478	valid_0's binary_logloss: 0.224646
Early stopping, best iteration is:
[402]	valid_0's auc: 0.960506	valid_0's binary_logloss: 0.224718
AUC: 0.9605062066761949
difference: 0.00015300711782306742
best iteration: 402
saved fold: 0

fold 1


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.957139	valid_0's binary_logloss: 0.237657
[200]	valid_0's auc: 0.958684	valid_0's binary_logloss: 0.228932
[300]	valid_0's auc: 0.959007	valid_0's binary_logloss: 0.228132
[400]	valid_0's auc: 0.959043	valid_0's binary_logloss: 0.228005
[500]	valid_0's auc: 0.959055	valid_0's binary_logloss: 0.227925
[600]	valid_0's auc: 0.959063	valid_0's binary_logloss: 0.227876
[700]	valid_0's auc: 0.959018	valid_0's binary_logloss: 0.2279
Early stopping, best iteration is:
[563]	valid_0's auc: 0.959069	valid_0's binary_logloss: 0.22789
AUC: 0.9590693897061866
difference: 0.0002324972197322861
best iteration: 563
saved fold: 1

fold 2


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.958764	valid_0's binary_logloss: 0.233317
[200]	valid_0's auc: 0.960444	valid_0's binary_logloss: 0.223964
[300]	valid_0's auc: 0.960894	valid_0's binary_logloss: 0.22283
[400]	valid_0's auc: 0.960984	valid_0's binary_logloss: 0.222561
[500]	valid_0's auc: 0.960995	valid_0's binary_logloss: 0.222497
Early stopping, best iteration is:
[420]	valid_0's auc: 0.961013	valid_0's binary_logloss: 0.22253
AUC: 0.9610125429716706
difference: 0.0001304816681211296
best iteration: 420
saved fold: 2

fold 3


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.958216	valid_0's binary_logloss: 0.235232
[200]	valid_0's auc: 0.959977	valid_0's binary_logloss: 0.226045
[300]	valid_0's auc: 0.960349	valid_0's binary_logloss: 0.224975
[400]	valid_0's auc: 0.960433	valid_0's binary_logloss: 0.224684
[500]	valid_0's auc: 0.960431	valid_0's binary_logloss: 0.224589
[600]	valid_0's auc: 0.96043	valid_0's binary_logloss: 0.224528
Early stopping, best iteration is:
[543]	valid_0's auc: 0.960465	valid_0's binary_logloss: 0.22453
AUC: 0.9604647204600252
difference: 0.00013980572633265353
best iteration: 543
saved fold: 3

fold 4


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.95846	valid_0's binary_logloss: 0.235292
[200]	valid_0's auc: 0.959942	valid_0's binary_logloss: 0.226158
[300]	valid_0's auc: 0.960262	valid_0's binary_logloss: 0.225159
[400]	valid_0's auc: 0.960261	valid_0's binary_logloss: 0.224995
[500]	valid_0's auc: 0.960285	valid_0's binary_logloss: 0.224875
[600]	valid_0's auc: 0.960323	valid_0's binary_logloss: 0.224772
[700]	valid_0's auc: 0.960237	valid_0's binary_logloss: 0.224776
Early stopping, best iteration is:
[600]	valid_0's auc: 0.960323	valid_0's binary_logloss: 0.224772
AUC: 0.9603229317236426
difference: 0.00011766556294101616
best iteration: 600
saved fold: 4


## Final predictions


In [7]:
missing_folds = [
    fold
    for fold in range(5)
    if not (oof_dir / f"proper_te_fold_{fold}.npy").exists()
    or not (test_predictions_dir / f"proper_te_fold_{fold}.npy").exists()
]

if missing_folds:
    raise FileNotFoundError(f"Missing folds: {missing_folds}")

oof_predictions = np.zeros(len(train_df), dtype="float32")
test_fold_predictions = []

for fold in range(5):
    valid_mask = train_df["fold"] == fold
    oof_predictions[valid_mask] = np.load(
        oof_dir / f"proper_te_fold_{fold}.npy"
    )
    test_fold_predictions.append(
        np.load(test_predictions_dir / f"proper_te_fold_{fold}.npy")
    )

test_predictions = np.mean(test_fold_predictions, axis=0).astype("float32")
oof_auc = roc_auc_score(train_df[target], oof_predictions)

np.save(oof_dir / "proper_te.npy", oof_predictions)
np.save(test_predictions_dir / "proper_te.npy", test_predictions)

final_metrics = {
    "parameters": parameters,
    "fold_metrics": fold_metrics,
    "mean_fold_auc": float(np.mean([fold_metrics[str(fold)]["auc"] for fold in range(5)])),
    "oof_auc": float(oof_auc),
}

with open(metrics_dir / "proper_te.json", "w", encoding="utf-8") as file:
    json.dump(final_metrics, file, indent=4, ensure_ascii=False)

submission = sample_submission.copy()
submission[target] = test_predictions
submission_path = submissions_dir / "proper_te_submission.csv"
submission.to_csv(submission_path, index=False)

print("OOF AUC:", oof_auc)
print("saved:", submission_path)
submission.head()

OOF AUC: 0.9602609586543622
saved: c:\Users\User\Desktop\Airline Satisfaction\submissions\proper_te_submission.csv


,id,satisfaction
0,699635,0.276020
1,699636,0.866279
2,699637,0.936316
3,699638,0.066135
4,699639,0.052806
